In [ ]:
# =====================================================
# Student Performance Predicition System
# GROUP 05
# 242UT2449P SEE CHWAN KAI
# 242UT24490 TEO JING AN
# 242UT2449Z KHO WEI CONG
# 242UT244B2 TEE KIAN HAO
# =====================================================
# Models used: ANN, DNN, Fuzzy System
# Metrics used: r2, MAE, RMSE (regression prediction)
# =====================================================

In [3]:
# install this library if not installed before
!pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 8.5 MB/s eta 0:00:00


In [4]:
# import libraries
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score # For regression evaluation

import tensorflow as tf
from tensorflow import keras
from keras import layers

# Fuzzy library
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [6]:
# Load dataset
CSV_PATH = "Final_Marks_Data.csv"
df = pd.read_csv(
    CSV_PATH,
    engine="python",
    on_bad_lines="skip" # skip the rows with data more than expected features
)

# Check the dataset content
print(f"Dataset loaded from: {CSV_PATH}")
print("\nFirst 5 rows of the DataFrame:")
print(df.head())

Dataset loaded from: Final_Marks_Data.csv

First 5 rows of the DataFrame:
  Student_ID  Attendance (%)  Internal Test 1 (out of 40)  \
0      S1000              84                           30   
1      S1001              91                           24   
2      S1002              73                           29   
3      S1003              80                           36   
4      S1004              84                           31   

   Internal Test 2 (out of 40)  Assignment Score (out of 10)  \
0                           36                             7   
1                           38                             6   
2                           26                             7   
3                           35                             7   
4                           37                             8   

   Daily Study Hours  Final Exam Marks (out of 100)  
0                  3                             72  
1                  3                             56  
2           

In [ ]:
# Check the dataset information
print("\nDataFrame Info:")
df.info()


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Student_ID                     2000 non-null   object
 1   Attendance (%)                 2000 non-null   int64 
 2   Internal Test 1 (out of 40)    2000 non-null   int64 
 3   Internal Test 2 (out of 40)    2000 non-null   int64 
 4   Assignment Score (out of 10)   2000 non-null   int64 
 5   Daily Study Hours              2000 non-null   int64 
 6   Final Exam Marks (out of 100)  2000 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 109.5+ KB


In [ ]:
# Check descriptive statistics
print("\nDescriptive Statistics:")
print(df.describe())

# all value is within accepted range


Descriptive Statistics:
       Attendance (%)  Internal Test 1 (out of 40)  \
count     2000.000000                  2000.000000   
mean        84.891500                    32.115500   
std          7.758855                     4.563504   
min         52.000000                    18.000000   
25%         80.000000                    29.000000   
50%         85.000000                    32.000000   
75%         90.000000                    35.000000   
max        100.000000                    40.000000   

       Internal Test 2 (out of 40)  Assignment Score (out of 10)  \
count                  2000.000000                   2000.000000   
mean                     32.464500                      7.507000   
std                       4.522827                      1.021015   
min                      16.000000                      4.000000   
25%                      29.000000                      7.000000   
50%                      33.000000                      8.000000   
75%         

In [ ]:
# Check if the dataset got missing value
print("Missing values per column:")
missing_values = df.isnull().sum()
print(missing_values)

if missing_values.sum() == 0:
    print("\nNo missing values found in the dataset. No action required.")
else:
    print("\nMissing values detected, performing missing values handling.")

    # remove rows with missing value
    df_before = df.shape[0]
    df = df.dropna()
    df_after = df.shape[0]

    # Check again missing values
    print(f"Rows before removing missing values: {df_before}")
    print(f"Rows after removing missing values: {df_after}")
    print(f"Removed {df_before - df_after} rows with missing values")

Missing values per column:
Student_ID                       0
Attendance (%)                   0
Internal Test 1 (out of 40)      0
Internal Test 2 (out of 40)      0
Assignment Score (out of 10)     0
Daily Study Hours                0
Final Exam Marks (out of 100)    0
dtype: int64

No missing values found in the dataset. No action required.


In [ ]:
# Data Engineering
df["internal_avg"] = (
    df["Internal Test 1 (out of 40)"] +
    df["Internal Test 2 (out of 40)"]
) / 2

df["internal_x_attendance"] = (
    df["internal_avg"] * df["Attendance (%)"]
)

df["study_x_internal"] = (
    df["Daily Study Hours"] * df["internal_avg"]
)

df["log_study"] = np.log1p(df["Daily Study Hours"])
df["sqrt_attendance"] = np.sqrt(df["Attendance (%)"])

In [ ]:
# assign target value to y
df["y"] = df["Final Exam Marks (out of 100)"].values

# Drop unused student ID and target variable
drop_cols = ["Student_ID", "Final Exam Marks (out of 100)"]
X = df.drop(columns=drop_cols + ["y"])
y = df["y"].values

# Train test split 8:2
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Validation split from train set 8:2
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=42
)

In [ ]:
# Identify numerical columns
numeric_cols = [
    "Attendance (%)", "Internal Test 1 (out of 40)", "Internal Test 2 (out of 40)",
    "Assignment Score (out of 10)", "Daily Study Hours", "internal_avg", "internal_x_attendance",
    "study_x_internal", "log_study", "sqrt_attendance",
]

# Convert numerical value using standardscaler
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="drop"
)

# Fit only on train
preprocess.fit(X_train)

X_train_p = preprocess.transform(X_train)
X_val_p = preprocess.transform(X_val)
X_test_p = preprocess.transform(X_test)

# Convert to dense arrays for Keras
# OneHotEncoder returns sparse by default
if hasattr(X_train_p, "toarray"):
    X_train_p = X_train_p.toarray()
    X_val_p = X_val_p.toarray()
    X_test_p = X_test_p.toarray()

input_dim = X_train_p.shape[1]

In [ ]:
# Regression based evaluation function for displaying the performance of the models
def evaluate_regression_model(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print("\n==============================")
    print(f"Model: {name}")
    print("==============================")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"R²   : {r2:.4f}")

    return {"rmse": rmse, "mae": mae, "r2": r2}

In [ ]:
# -------------------------------------------
# CI Model 1: ANN Model (one hidden layer)
# -------------------------------------------
def build_ann(input_dim, activation="relu", lr=1e-3, dropout=0.2, hidden_units=64):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden_units, activation=activation),
        layers.Dropout(dropout),
        layers.Dense(1)
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
# ------------------------------------------------
# CI Model 2: DNN Model (2 or more hidden layer)
# ------------------------------------------------
def build_dnn(input_dim, activation="relu", lr=1e-3, dropout=0.1, units=(256, 128, 64), l2_reg=1e-6):
    reg = keras.regularizers.l2(l2_reg)
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for u in units:
        model.add(layers.Dense(u, activation=activation, kernel_regularizer=reg))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout))

    model.add(layers.Dense(1))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
# configuring early_stop
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

def parameter_experiments():
    # Testing several hyperparameters on ANN and DNN
    experiments = [

        {"model": "ANN", "activation": "relu", "lr": 0.001, "dropout": 0.2, "hidden_units": 32}, # small hidden_units
        {"model": "ANN", "activation": "relu", "lr": 0.001, "dropout": 0.2,"hidden_units": 64}, # medium hidden_units
        {"model": "ANN", "activation": "relu", "lr": 0.001, "dropout": 0.2, "hidden_units": 128}, # large hidden_units
        {"model": "ANN", "activation": "tanh", "lr": 0.001, "dropout": 0.2, "hidden_units": 64}, # tanh activation

        {"model": "DNN", "activation": "relu", "lr": 0.001, "dropout": 0.2,"units": (128, 64, 32)},
        {"model": "DNN", "activation": "tanh", "lr": 0.001, "dropout": 0.35, "units": (128, 64, 32)}, # tanh activation
        {"model": "DNN", "activation": "relu", "lr": 0.002, "dropout": 0.3, "units": (128, 64, 32)}, # higher learning rate
        {"model": "DNN", "activation": "relu", "lr": 0.001, "dropout": 0.2, "units": (64, 32)}, # lesser depth
    ]

    results = []
    rng = np.random.default_rng(42)

    # Training
    for cfg in experiments:
        if cfg["model"] == "ANN":
            m = build_ann(
                input_dim,
                activation=cfg["activation"],
                lr=cfg["lr"],
                dropout=cfg["dropout"],
                hidden_units=cfg["hidden_units"]
            )
        else:
            m = build_dnn(
                input_dim,
                activation=cfg["activation"],
                lr=cfg["lr"],
                dropout=cfg["dropout"],
                units=cfg["units"]
            )

        m.fit(
            X_train_p, y_train,
            validation_data=(X_val_p, y_val),
            epochs=250,
            batch_size=32,
            callbacks=[early_stop],
            verbose=0
        )

        y_pred = m.predict(X_test_p, verbose=0).ravel()

        # Assign information to show in output
        if cfg["model"] == "ANN":
            model_name = (
                f"ANN hidden={cfg['hidden_units']} "
                f"act={cfg['activation']} "
                f"lr={cfg['lr']} "
                f"dropout={cfg['dropout']}"
            )
        else:
            model_name = (
                f"DNN units={cfg['units']} "
                f"act={cfg['activation']} "
                f"lr={cfg['lr']} "
                f"dropout={cfg['dropout']}"
            )

        scores = evaluate_regression_model(
            model_name,
            y_test,
            y_pred
        )
        scores.update(cfg)
        results.append(scores)

    return pd.DataFrame(results).sort_values("rmse")

exp_df = parameter_experiments()
print("\nTop experiment results by RMSE:")
print(exp_df[["model", "activation", "lr", "dropout", "rmse", "mae", "r2", "hidden_units", "units",]].head(10))


Model: ANN hidden=32 act=relu lr=0.001 dropout=0.2
RMSE : 4.7855
MAE  : 3.8256
R²   : 0.8171

Model: ANN hidden=64 act=relu lr=0.001 dropout=0.2
RMSE : 4.6957
MAE  : 3.7351
R²   : 0.8239

Model: ANN hidden=128 act=relu lr=0.001 dropout=0.2
RMSE : 4.6995
MAE  : 3.7305
R²   : 0.8237

Model: ANN hidden=64 act=tanh lr=0.001 dropout=0.2
RMSE : 4.7339
MAE  : 3.8006
R²   : 0.8211

Model: DNN units=(128, 64, 32) act=relu lr=0.001 dropout=0.2
RMSE : 4.7523
MAE  : 3.7522
R²   : 0.8197

Model: DNN units=(128, 64, 32) act=tanh lr=0.001 dropout=0.35
RMSE : 5.6730
MAE  : 4.4846
R²   : 0.7430

Model: DNN units=(128, 64, 32) act=relu lr=0.002 dropout=0.3
RMSE : 4.6882
MAE  : 3.7091
R²   : 0.8245

Model: DNN units=(64, 32) act=relu lr=0.001 dropout=0.2
RMSE : 4.7636
MAE  : 3.8019
R²   : 0.8188

Top experiment results by RMSE:
  model activation     lr  dropout      rmse       mae        r2  \
6   DNN       relu  0.002     0.30  4.688222  3.709142  0.824502   
1   ANN       relu  0.001     0.20  4.6957

In [ ]:
# -------------------------
# CI Model 3: Fuzzy System
# -------------------------

def build_fuzzy_system():
    # 4 inputs
    attendance = ctrl.Antecedent(np.arange(0, 101, 1), "attendance")
    internal   = ctrl.Antecedent(np.arange(0, 41, 1), "internal")
    assignment = ctrl.Antecedent(np.arange(0, 11, 1), "assignment")
    study      = ctrl.Antecedent(np.arange(0, 6, 1), "study")

    # Define output
    exam = ctrl.Consequent(np.arange(0, 101, 1), "exam")

    # Membership functions
    attendance["low"]  = fuzz.trimf(attendance.universe, [0, 60, 75])
    attendance["med"]  = fuzz.trimf(attendance.universe, [70, 85, 92])
    attendance["high"] = fuzz.trimf(attendance.universe, [88, 95, 100])

    internal["low"]  = fuzz.trimf(internal.universe, [0, 20, 27])
    internal["med"]  = fuzz.trimf(internal.universe, [25, 32, 36])
    internal["high"] = fuzz.trimf(internal.universe, [34, 38, 40])

    assignment["low"]  = fuzz.trimf(assignment.universe, [0, 5, 6.5])
    assignment["med"]  = fuzz.trimf(assignment.universe, [6, 7.5, 8.5])
    assignment["high"] = fuzz.trimf(assignment.universe, [8, 9, 10])

    study["low"]  = fuzz.trimf(study.universe, [0, 1.5, 2.5])
    study["med"]  = fuzz.trimf(study.universe, [2, 3, 4])
    study["high"] = fuzz.trimf(study.universe, [3.5, 4.5, 5])

    exam["low"]  = fuzz.trimf(exam.universe, [0, 40, 55])
    exam["med"]  = fuzz.trimf(exam.universe, [50, 65, 75])
    exam["high"] = fuzz.trimf(exam.universe, [70, 85, 100])

    # Define rules
    rules = [
        ctrl.Rule(internal["high"] & attendance["high"], exam["high"]),
        ctrl.Rule(internal["med"] & assignment["high"], exam["high"]),
        ctrl.Rule(internal["med"] & attendance["med"], exam["med"]),
        ctrl.Rule(internal["low"], exam["low"]),
        ctrl.Rule(study["high"] & assignment["med"], exam["med"]),
        ctrl.Rule(study["low"] & attendance["low"], exam["low"]),
    ]

    system = ctrl.ControlSystem(rules)
    return ctrl.ControlSystemSimulation(system)

def fuzzy_predict_regression(df_raw):
    preds = []

    for _, row in df_raw.iterrows():
        sim = build_fuzzy_system()

        sim.input["attendance"] = row["Attendance (%)"]
        sim.input["internal"] = (
            row["Internal Test 1 (out of 40)"] +
            row["Internal Test 2 (out of 40)"]
        ) / 2
        sim.input["assignment"] = row["Assignment Score (out of 10)"]
        sim.input["study"] = row["Daily Study Hours"]

        sim.compute()

        # Safe access fallback
        preds.append(sim.output.get("exam", 65.0))

    return np.array(preds)

fuzzy_pred = fuzzy_predict_regression(X_test)
evaluate_regression_model("Fuzzy System (Regression)", y_test, fuzzy_pred)


Model: Fuzzy System (Regression)
RMSE : 9.6785
MAE  : 7.6707
R²   : 0.2520


{'rmse': np.float64(9.678515035009578),
 'mae': 7.670699907114229,
 'r2': 0.2520492326525111}